# EDA 
1. Временные окна: проверим, есть ли события, которые произошли вне временного окна.
2. Дубликаты.
3. Связь признаков событий eid и event_name: проверим, эти признаки взаимозаменяемы или характеризуют разные события (т.е. оставлять один из этих признаков или оба).
4. Анализ платформ: нормализация и связь между платформой и событиями.
5. Порядок событий: отсортированы ли по времени.
6. Пропуски: оправданы т.к. есть поля, специфичные для событий, или же действительно есть пропущенные значения.
7. Валидация и бейзлайн из quickstart.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent / 'bot_detection_challenge'
sys.path.append(str(ROOT))
from metric import precision_at_recall

SEED = 42
DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']

train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=DATE_COLS)
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=DATE_COLS)
events_raw = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])

meta = pd.concat([train.assign(split='train'), test.assign(split='test')]).set_index('cookie_id')
train.shape, test.shape, events_raw.shape

((11091, 5), (4909, 4), (328905, 14))

## 1. Окно наблюдения

In [2]:
start = events_raw.cookie_id.map(meta.window_start_ts)
end = events_raw.cookie_id.map(meta.window_end_ts)
position = np.select([events_raw.event_ts < start, events_raw.event_ts >= end], ['before', 'after'], 'in')

ev = events_raw.assign(
    position=position,
    split=events_raw.cookie_id.map(meta.split),
    target=events_raw.cookie_id.map(meta.target),
)
pd.crosstab(ev.split, ev.position)

position,after,in
split,,
test,0,89690
train,40779,198436


In [3]:
ev_train = ev[ev.split == 'train']
pd.crosstab(ev_train.target, ev_train.position, normalize='index').round(2)

position,after,in
target,,
0.0,0.11,0.89
1.0,0.42,0.58


In [4]:
pd.crosstab(ev_train.event_name, [ev_train.position, ev_train.target])

position             after           in       
target                 0.0   1.0    0.0    1.0
event_name                                    
captcha_shown          958  6970      0      0
contact_chat_open      328   249   3404    376
contact_message_sent   161    97   1769    210
contact_phone_show     610   418   6361    685
favorite_add          1331   422  11317    720
item_view             7523  5208  62770  11568
login                  468   138   3693    207
photo_swipe           2447   986  21091   1860
search_results_view   6275  4329  52399   9318
seller_page_view      1105   756   9125   1563

События после окна есть только в train, причём у ботов их намного больше (42% против 11%). Это утечка, поэтому оставляем только события внутри окна.

`captcha_shown` встречается только после окна, как признак её использовать нельзя.

In [5]:
events = events_raw[position == 'in']
len(events_raw), len(events)

(328905, 288126)

## 2. Дубликаты

In [6]:
is_dup = events.duplicated()
print('полных дубликатов:', is_dup.sum())
is_dup.groupby(events.cookie_id.map(meta.target), dropna=False).mean().round(4)

полных дубликатов: 4293


cookie_id
0.0    0.0148
1.0    0.0152
NaN    0.0150
dtype: float64

Доля дубликатов одинакова у ботов, людей и в test. Это артефакт логирования, удаляем.

In [7]:
events = events.drop_duplicates()

## 3. Связь признаков событий eid и event_name

In [8]:
events.groupby('eid').event_name.unique()

eid
100     [search_results_view]
200               [item_view]
210             [photo_swipe]
220        [seller_page_view]
300      [contact_phone_show]
301       [contact_chat_open]
303    [contact_message_sent]
400            [favorite_add]
500                   [login]
Name: event_name, dtype: object

`eid` — код `event_name` (соответствие один к одному), удаляем его.

In [9]:
events = events.drop(columns='eid')

## 4. Анализ платформ

In [10]:
events.platform.value_counts()

platform
desktop    40159
WEB        39740
Web        39689
web        39310
ANDROID    37013
Android    36802
android    36741
ios         3600
iphone      3597
iOS         3593
IOS         3589
Name: count, dtype: int64

Приводим к трём платформам: убираем регистр, `desktop` → `web`, `iphone` → `ios`.

In [11]:
PLATFORM_MAP = {'android': 'android', 'ios': 'ios', 'iphone': 'ios', 'web': 'web', 'desktop': 'web'}
events['platform'] = events.platform.str.lower().map(PLATFORM_MAP)

events.platform.value_counts()

platform
web        158898
android    110556
ios         14379
Name: count, dtype: int64

Проверим, есть ли события, специфичные для той или иной платформы.

In [12]:
pd.crosstab(events.event_name, events.platform, normalize='columns').round(3)

platform,android,ios,web
event_name,,,
contact_chat_open,0.022,0.021,0.017
contact_message_sent,0.011,0.010,0.009
contact_phone_show,0.040,0.037,0.032
favorite_add,0.063,0.065,0.057
item_view,0.365,0.368,0.383
login,0.020,0.022,0.019
photo_swipe,0.121,0.120,0.110
search_results_view,0.302,0.306,0.319
seller_page_view,0.055,0.052,0.054


Набор событий на всех платформах одинаковый.

## 5. Порядок событий

In [13]:
events_raw.event_ts.is_monotonic_increasing

False

Файл не отсортирован по времени событий. Сортируем, т.к. некоторые признаки, например, интервалы между событиями, можно считать только после сортировки.

In [14]:
events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

## 6. Пропуски

In [15]:
optional = ['item_id', 'item_category', 'item_location', 'seller_type',
            'search_query', 'search_page', 'pointer_x', 'pointer_y']
events.groupby('event_name')[optional].agg(lambda s: s.notna().mean()).round(2)

,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
event_name,,,,,,,,
contact_chat_open,1.0,0.94,0.97,0.92,0.0,0.0,0.31,0.31
contact_message_sent,1.0,0.94,0.97,0.91,0.0,0.0,0.33,0.33
contact_phone_show,1.0,0.94,0.97,0.91,0.0,0.0,0.33,0.33
favorite_add,1.0,0.94,0.97,0.91,0.0,0.0,0.35,0.35
item_view,1.0,0.94,0.97,0.91,0.0,0.0,0.33,0.33
login,0.0,0.00,0.00,0.00,0.0,0.0,0.35,0.35
photo_swipe,1.0,0.94,0.97,0.91,0.0,0.0,0.34,0.34
search_results_view,0.0,0.94,0.97,0.00,1.0,1.0,0.33,0.33
seller_page_view,1.0,0.94,0.97,0.91,0.0,0.0,0.34,0.34


In [16]:
events.groupby('platform')[optional].agg(lambda s: s.notna().mean()).round(2)

,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
platform,,,,,,,,
android,0.68,0.92,0.95,0.62,0.30,0.30,0.00,0.00
ios,0.67,0.92,0.95,0.61,0.31,0.31,0.00,0.00
web,0.66,0.92,0.95,0.60,0.32,0.32,0.59,0.59


In [17]:
web = events[events.platform == 'web']
web.pointer_x.notna().groupby(web.cookie_id.map(meta.target)).mean().round(2)

cookie_id
0.0    0.66
1.0    0.33
Name: pointer_x, dtype: float64

Большинство пропусков структурные: у выдачи нет `item_id`, у карточек нет `search_*`, на мобильных нет курсора. Их не заполняем.

На web курсор есть у 66% событий людей и только у 33% событий ботов. Этот пропуск информативен и пригодится как признак.

## 7.1. Валидация

Test идёт после train по времени, поэтому валидируемся тоже по времени: 4 фолда по 2 дня, обучение на всех предыдущих днях.

Test отстоит от train на 1–7 дней, поэтому отдельно проверяем сплит с таким же горизонтом: обучение 06–12.04, валидация 13–19.04.

In [18]:
days = np.sort(train.window_start_ts.unique())
folds = []
for val_days in [days[6:8], days[8:10], days[10:12], days[12:14]]:
    tr = np.flatnonzero(train.window_start_ts < val_days[0])
    va = np.flatnonzero(train.window_start_ts.isin(val_days))
    folds.append((tr, va))

testlike_fold = [(np.flatnonzero(train.window_start_ts < '2026-04-13'),
                  np.flatnonzero(train.window_start_ts >= '2026-04-13'))]

for tr, va in folds:
    print(f'train {len(tr):5d} | val {len(va)} | ботов в val {train.target.iloc[va].sum()}')

train  5113 | val 1660 | ботов в val 120
train  6773 | val 1627 | ботов в val 135
train  8400 | val 1453 | ботов в val 126
train  9853 | val 1238 | ботов в val 94


В одном фолде около 100 ботов, поэтому метрику считаем по объединённым OOF-предсказаниям и добавляем бутстрап-интервал.

In [19]:
def cross_validate(model, X, y, folds):
    oof = np.full(len(y), np.nan)
    for tr, va in folds:
        model.fit(X.iloc[tr], y[tr])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]

    mask = ~np.isnan(oof)
    y_val, p_val = y[mask], oof[mask]

    rng = np.random.default_rng(SEED)
    boot = [precision_at_recall(y_val[i], p_val[i]) for i in rng.integers(0, len(y_val), (1000, len(y_val)))]

    return {
        'p_at_r70': round(precision_at_recall(y_val, p_val), 3),
        'ci95': np.percentile(boot, [2.5, 97.5]).round(3),
        'pr_auc': round(average_precision_score(y_val, p_val), 3),
        'roc_auc': round(roc_auc_score(y_val, p_val), 3),
    }

## 7.2. Baseline из quickstart
Также оценим влияние фильтра окна (т.к. без него получается утечка данных).

In [20]:
def quickstart_features(events):
    g = events.groupby('cookie_id')
    X = pd.DataFrame({'n_events': g.size(), 'item_nunique': g.item_id.nunique()})
    return X.reindex(train.cookie_id).fillna(0).reset_index(drop=True)


y = train.target.to_numpy()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED)

pd.DataFrame({
    'quickstart': cross_validate(rf, quickstart_features(events), y, folds),
    'quickstart, горизонт как в test': cross_validate(rf, quickstart_features(events), y, testlike_fold),
    'quickstart без фильтра окна': cross_validate(rf, quickstart_features(events_raw), y, folds),
}).T

,p_at_r70,ci95,pr_auc,roc_auc
quickstart,0.093,"[0.083, 0.107]",0.202,0.609
"quickstart, горизонт как в test",0.093,"[0.082, 0.104]",0.195,0.605
quickstart без фильтра окна,0.431,"[0.331, 0.537]",0.576,0.854


Доля ботов в валидации ~0.08 — это уровень случайного скора, quickstart даёт 0.09.

Без фильтра окна те же признаки дают 0.43, но весь этот прирост — утечка из событий после окна.